# Teacher Foundation Model — Training Pipeline

Baseline: `RESEARCH_BASELINE.md` (train on `master`; v1.0-baseline is the immutable release tag)  
Dataset: Google Drive → `MarketFoundation/storage/`

## Cell 1 — GPU Information

In [ ]:
import torch
import platform

print("=" * 60)
print("Python:", platform.python_version())
print("Torch :", torch.__version__)
print("CUDA  :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU   :", torch.cuda.get_device_name(0))
    print("VRAM  :", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

print("=" * 60)

## Cell 2 — Clone Repository & Checkout master

In [ ]:
import os
from pathlib import Path

# Always start from /content and remove any stale clone to avoid nested dirs
# if this cell is re-run (fixes /content/emptyu/emptyu/emptyu).
os.chdir("/content")
if Path("/content/emptyu").exists():
    !rm -rf /content/emptyu
!git clone https://github.com/sandeep999-cyber/emptyu.git
os.chdir("/content/emptyu")
# Train from master (docs + RESEARCH_BASELINE included). The v1.0-baseline
# tag is the immutable release anchor; git_commit is recorded in the run manifest.
!git checkout master

## Cell 2b — Verify Git Revision

In [ ]:
!git rev-parse HEAD
!git describe --tags
!git status --short

## Cell 3 — Install Dependencies

In [ ]:
!pip install -r requirements.txt

## Cell 3b — CUDA Re-check (fail-fast if torch was downgraded)

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "CUDA unavailable after dependency installation. "
    "PyTorch may have been replaced by a CPU-only build."
)

print("GPU:", torch.cuda.get_device_name(0))

## Cell 4 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Cell 4b — Locate Drive Storage

In [ ]:
from pathlib import Path

# The storage/ folder may be synced via "Computers" (My Computer) or "My Drive".
# Colab mounts them differently:
#   My Drive  → /content/drive/MyDrive/
#   Computers → /content/drive/My Computer/   (note the space)

CANDIDATES = [
    Path("/content/drive/MyDrive/storage"),           # My Drive sync
    Path("/content/drive/My Computer/storage"),       # Computers sync
    Path("/content/drive/MyDrive/MarketFoundation/storage"),
    Path("/content/drive/My Computer/MarketFoundation/storage"),
]

DRIVE_STORAGE = None
for candidate in CANDIDATES:
    if candidate.is_dir():
        DRIVE_STORAGE = candidate
        break

if DRIVE_STORAGE is None:
    print("Could not find storage/ on Drive. Listing contents:")
    for d in ["/content/drive/MyDrive", "/content/drive/My Computer"]:
        p = Path(d)
        if p.exists():
            print(f"\n{d}/")
            for item in sorted(p.iterdir()):
                print(f"  {item.name}")
    raise FileNotFoundError("Cannot locate storage/ on Google Drive")

# Derive the base MarketFoundation dir (parent of storage/)
DRIVE_BASE = DRIVE_STORAGE.parent
REPO = Path("/content/emptyu")

print(f"Found storage at: {DRIVE_STORAGE}")
print(f"Base directory:   {DRIVE_BASE}")
print(f"\nContents of {DRIVE_STORAGE}:")
for item in sorted(DRIVE_STORAGE.iterdir()):
    print(f"  {item.name}")

## Cell 5 — Copy Dataset to Local SSD & Symlink Outputs to Drive

In [ ]:
import shutil
from pathlib import Path

# DRIVE_BASE, DRIVE_STORAGE, and REPO are set by Cell 4b.

# --- Copy dataset to local SSD for performance & reliability ---
# storage/training: ~5 MB (manifests, fingerprints, DuckDB index)
# Only copy the futures 1m subset the trainer actually reads:
#   canonical/futures/<BTC|ETH|SOL>/{klines/1m, funding, open_interest, metadata}
# This is ~300 files / ~160 MB instead of all 5,060 files (~322 MB).
# Skip: exchange_info, other intervals (5m/15m/1h/4h/1d), spot market.

src_training = DRIVE_STORAGE / "training"
dst_training = REPO / "storage/training"
if dst_training.exists():
    shutil.rmtree(dst_training)
shutil.copytree(src_training, dst_training)
print(f"Copied {src_training} → {dst_training}")

TRAIN_SYMBOLS = ["BTCUSDT", "ETHUSDT", "SOLUSDT"]
SRC_SUBDIRS = ["klines/1m", "funding", "open_interest", "metadata"]

dst_canonical = REPO / "storage/canonical"
if dst_canonical.exists():
    shutil.rmtree(dst_canonical)

for sym in TRAIN_SYMBOLS:
    for sub in SRC_SUBDIRS:
        src = DRIVE_STORAGE / "canonical/futures" / sym / sub
        dst = dst_canonical / "futures" / sym / sub
        if src.is_dir():
            shutil.copytree(src, dst)
            size_kb = sum(p.stat().st_size for p in src.rglob("*") if p.is_file()) // 1024
            print(f"Copied futures/{sym}/{sub} ({size_kb} KB)")
        else:
            print(f"SKIPPED (missing on Drive): futures/{sym}/{sub}")

print(f"Copied canonical subset → {dst_canonical}")

# --- Symlink persistent output dirs → Drive ---
# These are checkpointed every epoch and must survive disconnects.
for name in ["models", "logs", "evaluation"]:
    dst = REPO / name
    src = DRIVE_BASE / name
    src.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        if dst.is_symlink() or dst.is_file():
            dst.unlink()
        else:
            shutil.rmtree(dst)
    dst.symlink_to(src)
    print(f"Symlinked {dst} → {src}")

# Verify local storage is in place
assert (REPO / "storage/training/index.duckdb").exists(), "Local training dir not copied"
assert (REPO / "storage/canonical").exists(), "Local canonical dir not copied"
print("\n✅ Data copied to local SSD; outputs symlinked to Drive")

## Cell 6 — Dataset Integrity Check

In [ ]:
import json
from pathlib import Path
import duckdb

REPO = Path("/content/emptyu")

# --- Assert fingerprint ---
fp = json.loads((REPO / "storage/training/dataset_fingerprint.json").read_text())
EXPECTED_FP = "328a7b67b070b95e47ba450452032a93dfa410431e0cf329de6a4ac7b5ae3875"
assert fp["fingerprint"] == EXPECTED_FP, f"Fingerprint mismatch: {fp['fingerprint']}"
assert fp["file_count"] == 510, f"File count mismatch: {fp['file_count']}"

# --- Assert manifest splits ---
manifest = json.loads((REPO / "storage/training/training_manifest_v1.json").read_text())
m = manifest["training_manifest"]
assert m["splits"]["train"]["symbols"] == ["BTCUSDT", "ETHUSDT"], "Train split mismatch"
assert m["splits"]["validation"]["symbols"] == ["SOLUSDT"], "Validation split mismatch"

# --- Assert DuckDB opens ---
conn = duckdb.connect(str(REPO / "storage/training/index.duckdb"))
conn.execute("SELECT count(*) FROM file_index_v1")
conn.close()

# --- Assert one parquet read succeeds ---
sample_dir = REPO / "storage/canonical/futures/BTCUSDT/klines/1m"
if sample_dir.exists():
    parquet_files = list(sample_dir.rglob("*.parquet"))
    assert len(parquet_files) > 0, "No parquet files found in sample dir"

print("✅ Dataset integrity verified")
print(f"  Fingerprint: {fp['fingerprint'][:16]}...")
print(f"  Files: {fp['file_count']}")
print(f"  Train: {m['splits']['train']['symbols']}")
print(f"  Validation: {m['splits']['validation']['symbols']}")
print(f"  Snapshot: {fp['dataset_versions']['snapshot']}")
print(f"  Alignment: {fp['dataset_versions']['alignment_version']}")
print(f"  Windowing: {fp['dataset_versions']['windowing_version']}")

## Cell 7 — Verify Snapshot

In [ ]:
from pathlib import Path

REPO = Path("/content/emptyu")
snapshot_dir = REPO / "storage/training/snapshots"
print(f"Snapshot dir: {snapshot_dir}")
print(f"Exists: {snapshot_dir.exists()}")

if snapshot_dir.exists():
    for p in sorted(snapshot_dir.iterdir()):
        print(f"  {p.name}")

## Cell 8 — Smoke Test

In [ ]:
import os
os.chdir("/content/emptyu")

!python -m src.training.train_teacher \
    --model-config configs/model_v1.yaml \
    --optimizer-config configs/optimizer_v1.yaml \
    --trainer-config configs/trainer_v1.yaml \
    --smoke

## Cell 8b — Experiment Summary

In [ ]:
import json
from pathlib import Path
import subprocess

REPO = Path("/content/emptyu")
fp = json.loads((REPO / "storage/training/dataset_fingerprint.json").read_text())
manifest = json.loads((REPO / "storage/training/training_manifest_v1.json").read_text())
m = manifest["training_manifest"]
git_hash = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=str(REPO)).decode().strip()[:8]
git_tag = subprocess.check_output(["git", "describe", "--tags"], cwd=str(REPO)).decode().strip()

print("=" * 60)
print("EXPERIMENT SUMMARY")
print("=" * 60)
print(f"Dataset fingerprint: {fp['fingerprint'][:16]}...")
print(f"Snapshot:          {fp['dataset_versions']['snapshot']}")
print(f"Train:             {m['splits']['train']['symbols']}")
print(f"Validation:        {m['splits']['validation']['symbols']}")
print(f"Model:             teacher_transformer_v1")
print(f"Objective:         Masked Market Modeling")
print(f"Commit:            {git_hash}")
print(f"Tag:               {git_tag}")
print("=" * 60)

## Cell 9 — 30-Epoch Scale Training

In [ ]:
import os

# Ensure we are in the repo root regardless of kernel state.
os.chdir("/content/emptyu")

print("=" * 60)
print("30-epoch scale training")
print()
print("Colab may disconnect before training finishes.")
print("Checkpoints are written every epoch.")
print("If disconnected, reconnect and use the Resume cell.")
print("=" * 60)

!python -m src.training.train_teacher \
    --model-config configs/model_v1.yaml \
    --optimizer-config configs/optimizer_v1.yaml \
    --trainer-config configs/trainer_v1_scale30.yaml

## Cell 10 — Resume Training (if needed)

In [ ]:
# Uncomment and fill in your run ID:
# !python -m src.training.train_teacher \
#     --model-config configs/model_v1.yaml \
#     --optimizer-config configs/optimizer_v1.yaml \
#     --trainer-config configs/trainer_v1.yaml \
#     --resume models/foundation/teacher_v1/<RUN_ID>

## Cell 11 — Find Latest Checkpoint

In [ ]:
import os
os.chdir("/content/emptyu")

from pathlib import Path

checkpoint_dirs = sorted(Path("models/foundation/teacher_v1").iterdir()) if Path("models/foundation/teacher_v1").exists() else []

if checkpoint_dirs:
    latest = checkpoint_dirs[-1]
    print(f"Latest checkpoint dir: {latest}")
    %env CHECKPOINT_DIR {latest}
else:
    print("No checkpoints found. Run training first.")

In [ ]:
import os
import sys
from pathlib import Path

print("=" * 60)
repo_ok = Path("/content/emptyu/.git").is_dir()
print(f"[{'OK' if repo_ok else 'MISSING'}] repo: /content/emptyu")
if repo_ok:
    os.chdir("/content/emptyu")
    sys.path.insert(0, "/content/emptyu")

data_ok = Path("/content/emptyu/storage/canonical").is_dir()
print(f"[{'OK' if data_ok else 'MISSING'}] data: storage/canonical (Cell 5)")

models_ok = Path("/content/emptyu/models").exists()
print(f"[{'OK' if models_ok else 'MISSING'}] symlink: models/ (Cell 5)")

ckpt_dirs = sorted(Path("models/foundation/teacher_v1").iterdir()) if (repo_ok and models_ok and Path("models/foundation/teacher_v1").exists()) else []
if ckpt_dirs:
    os.environ["CHECKPOINT_DIR"] = str(ckpt_dirs[-1])
    print(f"[OK] checkpoint: {os.environ['CHECKPOINT_DIR']}")
else:
    print("[MISSING] checkpoint dir under models/foundation/teacher_v1/")

print("=" * 60)
print("Recovery: after a Colab VM restart, re-run Cells 2 -> 4b -> 5 in order,")
print("then re-run this cell; all should read OK before running eval cells.")

## Cell 12 — Clustering Evaluation

In [ ]:
import os
import sys
if not os.path.isdir("/content/emptyu"):
    raise FileNotFoundError("/content/emptyu missing - run Cell 2 first")
os.chdir("/content/emptyu")
sys.path.insert(0, "/content/emptyu")
from pathlib import Path
if not os.environ.get("CHECKPOINT_DIR"):
    ckpt_dirs = sorted(Path("models/foundation/teacher_v1").iterdir())
    if not ckpt_dirs:
        raise FileNotFoundError("No checkpoints found - run training first")
    os.environ["CHECKPOINT_DIR"] = str(ckpt_dirs[-1])
    print(f"CHECKPOINT_DIR auto-set to {ckpt_dirs[-1]}")
else:
    print(f"Using CHECKPOINT_DIR={os.environ['CHECKPOINT_DIR']}")

!cd /content/emptyu && PYTHONPATH=/content/emptyu python -m src.evaluation.embedding.clustering \
    --checkpoint "$CHECKPOINT_DIR" \
    --split train \
    --pooling mean

## Cell 13 — Retrieval Evaluation

In [ ]:
import os
import sys
if not os.path.isdir("/content/emptyu"):
    raise FileNotFoundError("/content/emptyu missing - run Cell 2 first")
os.chdir("/content/emptyu")
sys.path.insert(0, "/content/emptyu")
from pathlib import Path
if not os.environ.get("CHECKPOINT_DIR"):
    ckpt_dirs = sorted(Path("models/foundation/teacher_v1").iterdir())
    if not ckpt_dirs:
        raise FileNotFoundError("No checkpoints found - run training first")
    os.environ["CHECKPOINT_DIR"] = str(ckpt_dirs[-1])
    print(f"CHECKPOINT_DIR auto-set to {ckpt_dirs[-1]}")
else:
    print(f"Using CHECKPOINT_DIR={os.environ['CHECKPOINT_DIR']}")

!cd /content/emptyu && PYTHONPATH=/content/emptyu python -m src.evaluation.embedding.retrieval \
    --checkpoint "$CHECKPOINT_DIR" \
    --split validation \
    --pooling mean

## Cell 14 — Temporal Consistency

In [ ]:
import os
import sys
if not os.path.isdir("/content/emptyu"):
    raise FileNotFoundError("/content/emptyu missing - run Cell 2 first")
os.chdir("/content/emptyu")
sys.path.insert(0, "/content/emptyu")
from pathlib import Path
if not os.environ.get("CHECKPOINT_DIR"):
    ckpt_dirs = sorted(Path("models/foundation/teacher_v1").iterdir())
    if not ckpt_dirs:
        raise FileNotFoundError("No checkpoints found - run training first")
    os.environ["CHECKPOINT_DIR"] = str(ckpt_dirs[-1])
    print(f"CHECKPOINT_DIR auto-set to {ckpt_dirs[-1]}")
else:
    print(f"Using CHECKPOINT_DIR={os.environ['CHECKPOINT_DIR']}")

!cd /content/emptyu && PYTHONPATH=/content/emptyu python -m src.evaluation.embedding.temporal_consistency \
    --checkpoint "$CHECKPOINT_DIR" \
    --split validation \
    --pooling mean

## Cell 15 — Linear Probe

In [ ]:
import os
import sys
if not os.path.isdir("/content/emptyu"):
    raise FileNotFoundError("/content/emptyu missing - run Cell 2 first")
os.chdir("/content/emptyu")
sys.path.insert(0, "/content/emptyu")
from pathlib import Path
if not os.environ.get("CHECKPOINT_DIR"):
    ckpt_dirs = sorted(Path("models/foundation/teacher_v1").iterdir())
    if not ckpt_dirs:
        raise FileNotFoundError("No checkpoints found - run training first")
    os.environ["CHECKPOINT_DIR"] = str(ckpt_dirs[-1])
    print(f"CHECKPOINT_DIR auto-set to {ckpt_dirs[-1]}")
else:
    print(f"Using CHECKPOINT_DIR={os.environ['CHECKPOINT_DIR']}")

import subprocess
subprocess.run(["git", "pull", "--ff-only"], cwd="/content/emptyu", check=True)
env = dict(os.environ, PYTHONPATH="/content/emptyu")
# One subprocess builds each split once and runs the encoder once per batch
# for cls/mean/attention, avoiding redundant work across pooling modes.
print("=== Linear probe: cls / mean / attention ===")
cmd = ["python", "-u", "-m", "src.evaluation.embedding.linear_probe",
       "--checkpoint", os.environ["CHECKPOINT_DIR"],
       "--pooling", "all", "--batch-size", "64"]
proc = subprocess.Popen(
    cmd, env=env, cwd="/content/emptyu", stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT, text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end="", flush=True)
returncode = proc.wait()
if returncode:
    raise subprocess.CalledProcessError(returncode, cmd)


## Cell 16 — Visualization

In [ ]:
import os
import sys
if not os.path.isdir("/content/emptyu"):
    raise FileNotFoundError("/content/emptyu missing - run Cell 2 first")
os.chdir("/content/emptyu")
sys.path.insert(0, "/content/emptyu")
from pathlib import Path
if not os.environ.get("CHECKPOINT_DIR"):
    ckpt_dirs = sorted(Path("models/foundation/teacher_v1").iterdir())
    if not ckpt_dirs:
        raise FileNotFoundError("No checkpoints found - run training first")
    os.environ["CHECKPOINT_DIR"] = str(ckpt_dirs[-1])
    print(f"CHECKPOINT_DIR auto-set to {ckpt_dirs[-1]}")
else:
    print(f"Using CHECKPOINT_DIR={os.environ['CHECKPOINT_DIR']}")

!cd /content/emptyu && PYTHONPATH=/content/emptyu python -m src.evaluation.embedding.visualization \
    --checkpoint "$CHECKPOINT_DIR" \
    --method pca \
    --pooling mean \
    --max-windows 5000 \
    --batch-size 64

## Cell 17 — §9 Baseline Harness

In [ ]:
import os
import subprocess
os.chdir("/content/emptyu")
env = dict(os.environ, PYTHONPATH="/content/emptyu")
cmd = ["python", "-u", "-m", "src.evaluation.baselines.runner",
       "--checkpoint", os.environ["CHECKPOINT_DIR"],
       "--pooling", "mean", "--max-windows", "1500",
       "--batch-size", "64", "--out", "evaluation/baselines"]
proc = subprocess.Popen(cmd, cwd="/content/emptyu", env=env,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
code = proc.wait()
if code:
    raise subprocess.CalledProcessError(code, cmd)


## Cell 18 — Archive Results & Sync DuckDB Back to Drive

In [ ]:
import shutil
from pathlib import Path

# DRIVE_BASE and DRIVE_STORAGE are normally set by Cell 4b; in a fresh
# session they may be undefined, so re-locate them here.
if "DRIVE_STORAGE" not in globals() or DRIVE_STORAGE is None:
    from google.colab import drive
    drive.mount("/content/drive")
    CANDIDATES = [
        Path("/content/drive/MyDrive/storage"),
        Path("/content/drive/My Computer/storage"),
        Path("/content/drive/MyDrive/MarketFoundation/storage"),
        Path("/content/drive/My Computer/MarketFoundation/storage"),
    ]
    DRIVE_STORAGE = next((c for c in CANDIDATES if c.is_dir()), None)
    if DRIVE_STORAGE is None:
        raise FileNotFoundError("Cannot locate storage/ on Drive; run Cells 4b and 5 first.")
    DRIVE_BASE = DRIVE_STORAGE.parent

REPO = Path("/content/emptyu")

# --- Archive evaluation results ---
archive_dest = DRIVE_BASE / "phase2_results"
shutil.make_archive(str(archive_dest), "zip", str(REPO / "evaluation"))
print(f"Results archived to {archive_dest}.zip")

# --- Sync updated DuckDB files back to Drive ---
# These were copied locally in Cell 5 for write reliability.
for db_name in ["index.duckdb", "experiment_registry.duckdb"]:
    local = REPO / "storage/training" / db_name
    remote = DRIVE_STORAGE / "training" / db_name
    if local.exists():
        shutil.copy2(local, remote)
        print(f"Synced {db_name} → Drive")

print("\n✅ Done")

## Optional — Save Environment Snapshot

In [ ]:
!pip freeze > environment.txt
!nvidia-smi